# 13 — Testing the Prototype and Improving It

**Milestone 4 · Task 14.** Test the recommendation prototype on sample
Kickstarter campaigns, check whether the predictions and recommendations are
understandable and useful, and improve the logic based on what testing shows.

Four kinds of test:

1. **Sample campaigns** across categories and risk levels
2. **Edge cases** designed to break things
3. **Real held-out campaigns** replayed through the engine, with the
   outcome each one actually had
4. A **log of what testing changed** — because it changed a good deal

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.prepare import load_splits
from src.recommend import get_recommender, campaign_from_row, BANDS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

rec = get_recommender("..")
test = load_splits("..")["random"][1]
print("Prototype ready.")

Prototype ready.


## 14.1 Ten sample campaigns

Invented but realistic, chosen to span categories, goal sizes, durations and
tier counts. None come from the dataset.

In [2]:
samples = {
    "Ambitious documentary":   dict(goal=40000, duration=60, reward_tiers=[10, 25, 50],
                                    category="Film & Video", subcategory="Documentary", state="CA"),
    "Board game, default 30d": dict(goal=8000, duration=30, reward_tiers=[10, 25, 50, 100],
                                    category="Games", subcategory="Board & Card Games", state="NY"),
    "Small illustration":      dict(goal=800, duration=28, reward_tiers=[5, 10, 25, 50, 75, 100, 150, 250],
                                    category="Art", subcategory="Illustration", state="OR"),
    "Hardware startup":        dict(goal=30000, duration=45, reward_tiers=[50, 150, 500],
                                    category="Technology", subcategory="Open Hardware", state="TX"),
    "Indie album":             dict(goal=3000, duration=35, reward_tiers=[5, 10, 25, 50],
                                    category="Music", subcategory="Indie Rock", state="CA"),
    "Big video game":          dict(goal=60000, duration=60, reward_tiers=[15, 30, 60, 100, 250, 1000],
                                    category="Games", subcategory="Video Games", state="WA"),
    "Children's book":         dict(goal=5000, duration=40, reward_tiers=[10, 20, 35, 60, 100, 150, 250],
                                    category="Publishing", subcategory="Children's Book", state="NY"),
    "Community theater":       dict(goal=2500, duration=25, reward_tiers=[10, 25, 50],
                                    category="Theater", subcategory="Theater", state="IL"),
    "Photography, one tier":   dict(goal=12000, duration=30, reward_tiers=[5],
                                    category="Photography", subcategory="Photography", state="TX"),
    "Food project":            dict(goal=20000, duration=45, reward_tiers=[1, 5, 10, 25, 50, 100, 200, 500, 1000],
                                    category="Food", subcategory="Food", state="CA"),
}
summary = rec.summary_frame(samples)
view = summary.assign(chance=(summary.chance * 100).round(0).astype(int).astype(str) + "%",
                      **{"plan chance": (summary["plan chance"] * 100).round(0).astype(int).astype(str) + "%",
                         "plan gain (pts)": summary["plan gain (pts)"].round(1)})
view

,campaign,chance,band,levers advised,plan chance,plan gain (pts)
0,Ambitious documentary,7%,High risk,"goal, duration, reward tiers",26%,19.2
1,"Board game, default 30d",12%,High risk,"goal, reward tiers",29%,16.7
2,Small illustration,85%,Favourable,reward tiers,88%,2.9
3,Hardware startup,11%,High risk,"goal, duration, reward tiers",40%,28.6
4,Indie album,32%,High risk,"goal, reward tiers",46%,13.8
5,Big video game,9%,High risk,"goal, duration, reward tiers",22%,13.5
6,Children's book,52%,Moderate risk,"goal, duration, reward tiers",70%,18.8
7,Community theater,58%,Favourable,"goal, reward tiers",77%,18.4
8,"Photography, one tier",6%,High risk,reward tiers,22%,15.5
9,Food project,14%,High risk,"goal, duration",23%,9.1


Things to check in this table:

- **The risk bands span the full range**, so the tool is not just saying
  "high risk" or "favourable" for everything.
- **No campaign got a plan that lowers its chance.**
- **Advice is driven mainly by reward tiers.** See the count below. That
  follows from the data — tiers is the strongest association — but it means
  the advice is less tailored than the table might suggest.
- **Plan gains are larger for riskier campaigns on average**, but not
  cleanly: the community theater project is already Favourable yet still
  gains about 18 points, because it offers only three tiers and the tier
  association is so strong. The small illustration, which already has eight
  tiers, gains about 3.

In [3]:
print(f"Bands represented: {sorted(summary['band'].unique())}")
print(f"Campaigns with no advice: {(summary['levers advised'] == 'none').sum()} of {len(summary)}")
print(f"Smallest plan gain: {summary['plan gain (pts)'].min():+.1f} pts  (never negative: "
      f"{bool((summary['plan gain (pts)'] >= 0).all())})")

lever_counts = pd.Series(
    [l.strip() for s_ in summary["levers advised"] for l in s_.split(",") if l.strip() and l.strip() != "none"]
).value_counts()
print("\nLevers advised, across the ten samples:")
print(lever_counts.to_string())
print("\nPlan gain by risk band (mean points):")
print(summary.groupby("band")["plan gain (pts)"].agg(["mean", "min", "max", "size"]).round(1).to_string())

Bands represented: ['Favourable', 'High risk', 'Moderate risk']
Campaigns with no advice: 0 of 10
Smallest plan gain: +2.9 pts  (never negative: True)

Levers advised, across the ten samples:
reward tiers    9
goal            8
duration        5

Plan gain by risk band (mean points):
               mean   min   max  size
band                                 
Favourable     10.7   2.9  18.4     2
High risk      16.6   9.1  28.6     7
Moderate risk  18.8  18.8  18.8     1


### Two full reports

A campaign left at the platform default (checks the 30-day rule) and one
that is already strong (checks that the tool does not oversell changes).

In [4]:
print(rec.report(samples["Board game, default 30d"]))

CAMPAIGN ASSESSMENT
  Games / Board & Card Games - NY - goal $8,000 - 30 days - 4 reward tiers ($10,$25,$50,$100)

Predicted chance of reaching the goal: 12%   [High risk]   (final model: Random Forest (tuned + calibrated))
  Held-out campaigns scored 7%-17% actually succeeded 10% of the time (n=674).

Where you stand (Board & Card Games, 358 campaigns)
  Goal: $8,000 is at the 59th percentile (median $5,500).
  Campaigns with a similar goal (half to double yours): 53% succeeded (n=204).
  Successful campaigns here typically ran 35 days with 10 reward tiers.

What you could change (estimated effect on your chance)
  [goal        ] Lower the goal to $7,200: 13% (+1.6 pts)
  [goal        ] Lower the goal to $6,000: 17% (+5.1 pts)  <- recommended
  [goal        ] Lower the goal to $4,800: 20% (+8.6 pts)
  [goal        ] Lower the goal to $4,000: 24% (+11.8 pts)
  [reward tiers] Offer 6 reward tiers ($10,$15,$25,$50,$75,$100): 24% (+11.8 pts)  <- recommended
  [reward tiers] Offer 8 reward

In [5]:
print(rec.report(samples["Small illustration"]))

CAMPAIGN ASSESSMENT
  Art / Illustration - OR - goal $800 - 28 days - 8 reward tiers ($5,$10,$25,$50,$75,$100,$150,$250)

Predicted chance of reaching the goal: 85%   [Favourable]   (final model: Random Forest (tuned + calibrated))
  Held-out campaigns scored 80%-90% actually succeeded 86% of the time (n=1,043).

Where you stand (Illustration, 133 campaigns)
  Goal: $800 is at the 13th percentile (median $2,600).
  Campaigns with a similar goal (half to double yours): 62% succeeded (n=34).
  Successful campaigns here typically ran 33 days with 8 reward tiers.

What you could change (estimated effect on your chance)
  [goal        ] Lower the goal to $720: 86% (+0.6 pts)
  [goal        ] Lower the goal to $600: 86% (+1.5 pts)
  [reward tiers] Offer 10 reward tiers ($5,$10,$15,$25,$40,$50,$75,$100,$150,$250): 88% (+2.9 pts)  <- recommended

Combined plan: 85% -> 88% (+2.9 pts)
  - Offer 10 reward tiers ($5,$10,$15,$25,$40,$50,$75,$100,$150,$250)
  Goal x probability: $680 -> $703  (a flo

## 14.2 Edge cases

Inputs chosen to find failures. The criterion is not "does it give advice"
but "does it behave sensibly and never crash".

In [6]:
base = samples["Ambitious documentary"]

def run_case(label, campaign):
    try:
        a = rec.assess(campaign)
        levers = a.options.loc[a.options["recommended"], "lever"].tolist() if len(a.options) else []
        return {"case": label, "result": "ok", "chance": a.probability, "band": a.band,
                "advised": ", ".join(levers) or "none",
                "plan gain (pts)": a.plan["gain_pp"] if a.plan else 0.0,
                "notes": len(a.notes)}
    except Exception as e:                       # noqa: BLE001 - we want to see any failure
        return {"case": label, "result": f"ERROR {type(e).__name__}: {str(e)[:60]}",
                "chance": np.nan, "band": "", "advised": "", "plan gain (pts)": np.nan, "notes": 0}

many = [1, 2, 3, 5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100, 125, 150, 200, 250,
        300, 400, 500, 750, 1000, 1500, 2000]
cases = {
    "duration 30.3 (inside the default window)": {**base, "duration": 30.3},
    "duration exactly 1 day": {**base, "duration": 1},
    "duration exactly 60 days": {**base, "duration": 60},
    "single reward tier": {**base, "reward_tiers": [25]},
    "25 reward tiers": {**base, "reward_tiers": many},
    "duplicate tier prices": {**base, "reward_tiers": [10, 10, 25, 25]},
    "tiny goal ($50) in a larger subcategory": {**base, "goal": 50},
    "huge goal ($5,000,000)": {**base, "goal": 5_000_000},
    "goal below the subcategory floor": {**samples["Small illustration"], "goal": 50},
    "all optional fields supplied": {**base, "city": "Los Angeles", "name": "A Film", "launch_date": "2012-01-15"},
    "state is DC": {**base, "state": "DC"},
    "launch date far outside training years": {**base, "launch_date": "2026-05-26"},
}
edge = pd.DataFrame([run_case(k, v) for k, v in cases.items()])
edge.assign(chance=edge.chance.map(lambda v: f"{v:.1%}" if pd.notna(v) else ""),
            **{"plan gain (pts)": edge["plan gain (pts)"].round(1)})

,case,result,chance,band,advised,plan gain (pts),notes
0,duration 30.3 (inside the default window),ok,6.7%,High risk,"goal, reward tiers",12.9,3
1,duration exactly 1 day,ok,8.6%,High risk,"goal, reward tiers",13.2,3
2,duration exactly 60 days,ok,6.6%,High risk,"goal, duration, reward tiers",19.2,2
3,single reward tier,ok,6.0%,High risk,"goal, reward tiers",27.0,2
4,25 reward tiers,ok,39.5%,Moderate risk,"goal, duration",8.1,2
5,duplicate tier prices,ok,6.5%,High risk,"goal, duration, reward tiers",13.5,2
6,tiny goal ($50) in a larger subcategory,ok,63.4%,Favourable,"duration, reward tiers",15.2,1
7,"huge goal ($5,000,000)",ok,6.3%,High risk,none,0.0,1
8,goal below the subcategory floor,ok,84.1%,Favourable,none,0.0,2
9,all optional fields supplied,ok,8.0%,High risk,"goal, duration, reward tiers",18.1,2


In [7]:
print(f"Cases run: {len(edge)}   crashed: {(~edge['result'].eq('ok')).sum()}")

Cases run: 12   crashed: 0


In [8]:
print("What the report says for the $5,000,000 goal:\n")
print(rec.assess({**base, "goal": 5_000_000}).notes[0])

What the report says for the $5,000,000 goal:

Your chance is very low and none of the modelled changes lifts it by even 2 points. Your goal is at the 100th percentile for Documentary (median $8,000). The Random Forest cannot tell campaigns this far out from merely large ones, so read its 6% as an upper bound (the Logistic Regression puts it at 0.1%). These levers cannot rescue a goal that far from what comparable campaigns ask for.


### Launch date outside the training years

The model has only seen 2009-2012, and launch year enters as a number.
Extrapolating that to 2026 would be meaningless, so the prototype clips the
year to the trained range. Check it: a 2026 date and a 2012 date with the
same month and weekday should give the identical answer.

In [9]:
p_2026 = rec.probability({**base, "launch_date": "2026-05-26"})   # a Tuesday in May
p_2012 = rec.probability({**base, "launch_date": "2012-05-29"})   # a Tuesday in May
print(f"2026-05-26 (Tue, May): {p_2026:.4%}")
print(f"2012-05-29 (Tue, May): {p_2012:.4%}")
print(f"Identical: {abs(p_2026 - p_2012) < 1e-12}")

2026-05-26 (Tue, May): 6.6279%
2012-05-29 (Tue, May): 6.6279%
Identical: True


## 14.3 Real campaigns, replayed

Held-out campaigns the model never trained on, with the outcome each one
actually had. This is the closest thing to a real test the data allows —
**and it has a hard limit**: it shows what the engine would have *said*, not
whether following that advice would have *worked*. The campaign did not
change its goal, so the data cannot tell us what would have happened.

In [10]:
tt = test.assign(p=rec.holdout_p)
tt = tt[tt["duration_days"] <= 60]

picks = [
    ("failed, scored high risk",       tt[tt.success == 0].nsmallest(3, "p")),
    ("funded, scored high risk",       tt[tt.success == 1].nsmallest(2, "p")),
    ("funded, scored favourable",      tt[(tt.success == 1) & (tt.p > 0.75)].sample(2, random_state=0)),
    ("failed, scored favourable",      tt[tt.success == 0].nlargest(1, "p")),
]
rows = []
for kind, df in picks:
    for _, r in df.iterrows():
        a = rec.assess(campaign_from_row(r))
        levers = a.options.loc[a.options["recommended"], "lever"].tolist() if len(a.options) else []
        rows.append({"what happened": kind, "project": str(r["name"])[:34],
                     "subcategory": r["subcategory"], "goal": f"${r['goal']:,.0f}",
                     "days": round(r["duration_days"]), "tiers": int(r["levels"]),
                     "outcome": "funded" if r["success"] else "failed",
                     "model chance": f"{a.probability:.0%}",
                     "plan chance": f"{(a.plan['probability'] if a.plan else a.probability):.0%}",
                     "advised": ", ".join(levers) or "none"})
replay = pd.DataFrame(rows)
replay

,what happened,project,subcategory,goal,days,tiers,outcome,model chance,plan chance,advised
0,"failed, scored high risk",Abby Diamond- Blind Girl Detective,Animation,"$57,000",30,1,failed,6%,11%,reward tiers
1,"failed, scored high risk",Prince Adventures: The Anime Serie,Animation,"$300,000",60,7,failed,6%,6%,none
2,"failed, scored high risk",Magic Town trilogy publication,Fiction,"$25,000",30,2,failed,6%,20%,"goal, reward tiers"
3,"funded, scored high risk",Beyond the Mask Movie,Narrative Film,"$20,000",47,4,funded,7%,21%,"goal, duration, reward tiers"
4,"funded, scored high risk",Neil Gaiman's The Price,Animation,"$150,000",32,9,funded,8%,10%,goal
5,"funded, scored favourable","Numerology Ology, a new record by",Music,"$6,000",33,13,funded,86%,88%,goal
6,"funded, scored favourable",First Law of Mad Science Issue 1,Comics,"$2,200",32,12,funded,84%,87%,goal
7,"failed, scored favourable",Think Tank Theater Company: Test P,Theater,"$1,200",26,7,failed,90%,94%,"goal, reward tiers"


**How to read this honestly.**

- Campaigns that **failed after being scored high risk** are the model
  working as intended. Look at their plan chance, though: even the best plan
  leaves every one of them in the High-risk band, and the $300,000 animation
  gets no advice at all. Their goals are far beyond what the three levers can
  fix, and the tool does not pretend otherwise.
- Campaigns that **funded despite a high-risk score** are the model's
  misses, and they are why a probability is not a verdict. At 19% a campaign
  in the High-risk band still succeeds roughly one time in five.
- The campaign that **failed despite a favourable score** is the same point
  in reverse.

None of this shows that the advice would have *changed* an outcome.

## 14.4 What testing changed

Testing the prototype changed it substantially. Every row below was found
by running it, not planned in advance. Rows 9, 10 and 14 were mistakes in
earlier September work that building the recommender exposed; row 12 was a
false statement in the tool's own output; row 15 came from switching the
final model to the Random Forest.

In [11]:
pd.DataFrame([
    (1, "Report said \"91% percentile\"", "Wording fixed to \"91st percentile\"."),
    (2, "A campaign scored 88% was compared with its whole band's 74% success rate, which made a good prediction look wrong", "Replaced with a **local** rate: held-out campaigns scored near that probability."),
    (3, "Suggested reward tiers for narrow ranges came out as $12, $13, $15, $17 - tiers no creator would offer", "Tiers now fill the widest price gaps and snap to believable amounts; if no room is left, a higher tier is added instead."),
    (4, "Moving a campaign one day either side of 30 changed its predicted chance by about 5.5 points", "Traced to the model reacting to the \"left at default\" flag. The engine now never advises from or to 30 days and says so."),
    (5, "Tier advice could read as a guarantee", "Added a note that the tier effect is the one most likely to be partly selection, and an \"upper estimate\" line on every report."),
    (6, "A lower goal was shown as pure upside", "Plans now state what a lower goal gives up, and show goal x probability as a floor on expected funds."),
    (7, "Typing in `updates` or `backers` was silently ignored", "Now refused with an explanation. The tool cannot be misused into leaking."),
    (8, "The entry-price note stated a model behaviour as fixed text", "Now computed for each campaign, then rewritten after the real cause was found."),
    (9, "**September claim: \"the 30-day default outperforms\"**", "False - 30-day campaigns succeed *less* (52.9% vs 55.5%). Corrected here and flagged for the README and notebook 04."),
    (10, "**September claim: \"a low entry price helps\" / \"add a $5 tier\"**", "A tier-count effect. High-entry campaigns offer a median of 3 tiers against 8. With goal and tier count held fixed the gap shrinks from -23 points to a few. No entry-price lever was built."),
    (11, "The report said city, title and date were \"ignored unless supplied\"", "Measured instead: leaving out the city moves predictions ~5 points on average. Wording corrected; the effect on the *advice* is smaller (identical advice for 90% of campaigns)."),
    (12, "A $5,000,000 goal (chance 0.1%) got the note \"already close to what comparable campaigns do\" - plainly false, and an unverified reassurance", "A point-based threshold cannot register gains near zero. The note now depends on how low the chance is, states the goal's percentile and the subcategory median, and says the levers cannot rescue a goal that far out."),
    (13, "A project with nine reward tiers was told to add a tenth", "Tier advice must now add at least two tiers (`MIN_TIER_ADDITION`); one more is noise, not advice."),
    (14, "**September claim: \"location is a modest signal\"** - found while measuring how much the optional city field matters", "False. Success runs from 65% (NY) to 38% (FL) across states, a 27-point spread that survives removing goal size, and from 43% to 62% across city-size quartiles. Corrected in notebook 10 section 11.6; the city feature is why a fairness check was added to notebook 14."),
    (15, "After switching the final model to the Random Forest, re-scoring what-ifs with the forest made \"lower your goal\" *reduce* the chance for 11-19% of campaigns", "Scoring stays with the Random Forest; the effect of each change is estimated by the Logistic Regression advisor and added in percentage points (notebook 10 section 11.8), so advice always points the same way. A first version used a log-odds shift; testing on extreme goals (a $5,000,000 campaign was promised +20 points) showed it inflated gains where the forest cannot extrapolate, so it was replaced."),
], columns=["#", "What testing revealed", "What changed"])

,#,What testing revealed,What changed
0,1,"Report said ""91% percentile""","Wording fixed to ""91st percentile""."
1,2,A campaign scored 88% was compared with its whole band's...,Replaced with a **local** rate: held-out campaigns score...
2,3,Suggested reward tiers for narrow ranges came out as $12...,Tiers now fill the widest price gaps and snap to believa...
3,4,Moving a campaign one day either side of 30 changed its ...,"Traced to the model reacting to the ""left at default"" fl..."
4,5,Tier advice could read as a guarantee,Added a note that the tier effect is the one most likely...
5,6,A lower goal was shown as pure upside,"Plans now state what a lower goal gives up, and show goa..."
6,7,Typing in `updates` or `backers` was silently ignored,Now refused with an explanation. The tool cannot be misu...
7,8,The entry-price note stated a model behaviour as fixed text,"Now computed for each campaign, then rewritten after the..."
8,9,"**September claim: ""the 30-day default outperforms""**",False - 30-day campaigns succeed *less* (52.9% vs 55.5%)...
9,10,"**September claim: ""a low entry price helps"" / ""add a $5...",A tier-count effect. High-entry campaigns offer a median...


## 14.5 Is it understandable and useful?

The brief asks whether the output is "understandable and useful". Honest
answer on each:

**Understandable** — the report is plain text with a fixed order: chance,
where you stand, options, plan, caveats. Every number is in points or
percentages, there is no model jargon, and each option shows its effect.
*Not tested:* with anyone other than the team.

**Useful** — the advice is concrete (a specific goal, a specific duration, a
specific tier list) and ranked. *Limits:* it can only speak to goal,
duration and reward tiers; it knows nothing about the project itself; and
the tier advice in particular should be read as "campaigns with more tiers
did better", not "adding tiers will cause success".

**What this testing did not cover**

- **No real creators.** Nobody outside the team has used it, so "useful" is
  our judgement, not theirs.
- **No causal test.** The data has no campaign that changed one thing and
  was observed again.
- **No present-day data.** Everything is 2009-2012.

In [12]:
summary.round(4).to_csv("../reports/m4_sample_assessments.csv", index=False)
edge.round(4).to_csv("../reports/m4_edge_cases.csv", index=False)
replay.to_csv("../reports/m4_replayed_campaigns.csv", index=False)
print("Wrote m4_sample_assessments.csv, m4_edge_cases.csv, m4_replayed_campaigns.csv")

Wrote m4_sample_assessments.csv, m4_edge_cases.csv, m4_replayed_campaigns.csv


## Task 14 — Summary

- **Ten sample campaigns** span the risk range. Plan gains are largest for
  high-risk campaigns on average and no plan lowers a campaign's chance, but
  most of the advice is about reward tiers.
- **Twelve edge cases, none crashed.** The 30-day rule, the one-tier case,
  the 25-tier case, tiny and huge goals and out-of-range launch dates all
  behave sensibly.
- **Real held-out campaigns replayed**, with a clear statement of what that
  can and cannot show.
- **Fifteen improvements** came out of testing, including corrections to
  four September claims and one false statement in the tool's own output.

**Next:** `14_validation.ipynb` is the final check that the model and the
recommendations are consistent, reasonable and based only on launch-time
information.